# Notebook D2 -- GATE ONLY (verify gate for Qwen2.5-VL-3B, InternVL3-2B, Qwen2.5-VL-7B)
Runs **only** `mech_e3.py --parts verify` for the three models that crashed in notebook D. No experiment part runs.
Goal: find out in ~20-30 min (a GUESS; real timings are printed in the logs) whether the machinery is sound on real weights
and 2 GPUs, BEFORE launching the full re-run (notebook `notebook_D3_E3_rerun_no8b.ipynb`).

What changed since the FIRST D2 run (see the deviations log in `PREREG_MECHANISM.md`, entry 2026-11-06 (second entry of that day)):
* **Bug B fix (attention blocking had no effect on InternVL3-2B / Qwen-7B):** the stock *eager* attention multiplies q.k^T in fp16;
  fused SDPA accumulates in fp32. If a head overflows, eager gives NaN/garbage while default attention is fine -- which fits
  G8 = 0.0, G9 = 0/20, G7b ~ 0/20 (a NaN difference was silently counted as "no change"). Inside our hook context the eager
  function is now an fp32-accumulating copy (same math), AND the gate now checks that eager is really active on every layer
  (G8a, prints the per-layer implementation), that eager logits are finite and close to the default path (G8c), and counts NaN
  as a failure (G7d; G7b/G7c).
* **Bug A (Qwen no-cache loop vs cached `generate`):** `rope_deltas` is reset before every no-cache forward (defensive), and
  new diagnostics separate the possible causes: G10 (cached incremental decode vs full forward on the same tokens, max logit diff
  + argmax agreement), the no-cache greedy loop is compared to cached generation per trial, and every G2-own disagreement is
  classified (off-grid free minute / near-tie < 0.5 nat / clear disagreement) and listed by prefix class.
* Thresholds are unchanged. G2 is still gated by the own-prefix agreement >= 0.95.

## Inputs
| Dataset | Contents at the dataset ROOT |
|---|---|
| **clock-code** (new VERSION of the existing one) | `adapters.py`, `clocks.py`, `eval_behavior.py`, `text_only_v2.py`, `conversion_test.py`, `cap_check.py`, `mech_lib.py` (UPDATED), `mech_e3.py` (UPDATED) |

No prior-results dataset is needed for the gate. **Settings:** GPU T4 x2, Internet ON, Save & Run All (or run cells one by one).
**Output:** `mech_D2_gate_results.zip` -- send it back; cell 6 also prints the gate table.


In [ ]:
# CELL 1 -- GPU CHECK: stop the notebook right here if CUDA is not available
import torch, subprocess

MIN_GPUS = 2
if not torch.cuda.is_available():
    raise RuntimeError("STOP: CUDA is not available. Settings -> Accelerator -> 'GPU T4 x2', then Save & Run All again.")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
if n_gpu < MIN_GPUS:
    raise RuntimeError(f"STOP: this notebook needs {MIN_GPUS} GPU(s) but only {n_gpu} visible. Choose 'GPU T4 x2'.")
subprocess.run(["nvidia-smi"])
print("GPU check passed.")


In [ ]:
# CELL 2 -- COPY INPUTS from the attached Kaggle datasets into a clean working folder
import os, shutil, sys, time, json, subprocess, zipfile, datetime, hashlib

T_START = time.time()
INPUT, WORK = "/kaggle/input", "/kaggle/working"
RUN = f"{WORK}/run"
OUT_ZIP = f"{WORK}/mech_D2_gate_results.zip"
CODE_FILES = ["adapters.py", "clocks.py", "eval_behavior.py", "text_only_v2.py", "conversion_test.py", "cap_check.py", "mech_lib.py", "mech_e3.py"]

shutil.rmtree(RUN, ignore_errors=True)
os.makedirs(f"{RUN}/logs")

def find_root(marker, maxdepth=4):
    """Directory under /kaggle/input that contains `marker` (a relative path). Dataset slugs don't matter."""
    base = INPUT.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(INPUT):
        if dirpath.count("/") - base > maxdepth:
            dirnames[:] = []
            continue
        if os.path.exists(os.path.join(dirpath, marker)):
            return dirpath
    raise FileNotFoundError(f"Could not find '{marker}' under {INPUT}. Attach the dataset that contains it "
                            f"(see the checklist at the top of this notebook).")

code_root = find_root("adapters.py")
for f in CODE_FILES:
    shutil.copy(os.path.join(code_root, f), RUN)
print("code from:", code_root)

print("inputs copied OK.")
du = shutil.disk_usage("/"); print(f"disk: {du.free / 2**30:.1f} GiB free of {du.total / 2**30:.1f} GiB")


In [ ]:
# CELL 3 -- ENVIRONMENT: env vars, versions, internet check (model weights are downloaded from Hugging Face)
os.environ.update({"PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "60", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                   "TOKENIZERS_PARALLELISM": "false", "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"})
r = subprocess.run([sys.executable, "-c",
    "import torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL, sys;"
    "print('python', sys.version.split()[0]);"
    "[print(m.__name__, m.__version__) for m in (torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL)];"
    "print('cuda', torch.version.cuda, '| gpus', torch.cuda.device_count())"],
    capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])
assert r.returncode == 0, "a required package is missing -- see the error above"
open(f"{RUN}/env.txt", "w").write(r.stdout + "\nstarted " + datetime.datetime.utcnow().isoformat() + "Z\n")
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=20)
    print("internet OK (huggingface.co reachable)")
except Exception as e:
    raise RuntimeError(f"STOP: huggingface.co not reachable ({e}). Settings -> Internet -> ON.")


In [ ]:
# CELL 4 -- THE STEPS: (name, command, guessed_minutes)
M3B, A3B = "Qwen/Qwen2.5-VL-3B-Instruct", "qwen2.5-vl-3b"
M2B, A2B = "OpenGVLab/InternVL3-2B-hf", "internvl3-2b"
M7B, A7B = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen2.5-vl-7b"
M8B, A8B = "OpenGVLab/InternVL3-8B-hf", "internvl3-8b"
DEADLINE_H = 2.5      # do not START a new step after this many hours
HARD_END_H = 3.0      # do not start a step whose guessed duration would end after this
MAXTOK_8B = "128"      # was 32 (notebook B) then 64 (notebook C)

def e3(model_id, adapter, parts, extra=()):
    return ["python", "mech_e3.py", "--model_id", model_id, "--adapter", adapter, "--parts", parts, *extra]

def purge_cache(*repos):
    code = ("import shutil,os,glob;"
            "[shutil.rmtree(p, ignore_errors=True) for r in %r for p in glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--'+r))]" % (list(repos),))
    return ["python", "-c", code]

STEPS = [
    ("01_q3b_gate_only",  e3(M3B, A3B, "verify"), 12),
    ("02_i2b_gate_only",  e3(M2B, A2B, "verify"), 10),
    ("03_purge_3b_2b_cache", purge_cache("Qwen--Qwen2.5-VL-3B-Instruct", "OpenGVLab--InternVL3-2B-hf"), 1),
    ("04_q7b_gate_only",  e3(M7B, A7B, "verify"), 18),
]
for name, cmd, est in STEPS:
    print(f"{name:<32} ~{est:>3} min (guess)  {' '.join(cmd)[:110]}")
print(f"guessed total: {sum(e for _, _, e in STEPS) / 60:.1f} h")


In [ ]:
# CELL 5 -- RUN every step. A failing step is recorded and the notebook CONTINUES with the next one.
EXPECTED = ['results/mech_e3/qwen2.5-vl-3b/verify.json', 'results/mech_e3/qwen2.5-vl-3b/verify_g2_trials.csv', 'results/mech_e3/internvl3-2b/verify.json', 'results/mech_e3/internvl3-2b/verify_g2_trials.csv', 'results/mech_e3/qwen2.5-vl-7b/verify.json', 'results/mech_e3/qwen2.5-vl-7b/verify_g2_trials.csv']        # files that should exist at the end (relative to the run folder)
status = []

def make_zip():
    """Whitelisted snapshot of everything worth keeping; refreshed after EVERY step."""
    tmp = OUT_ZIP + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("results", "logs"):
            for dp, _, fns in os.walk(f"{RUN}/{sub}"):
                for fn in fns:
                    full = os.path.join(dp, fn); z.write(full, os.path.relpath(full, RUN))
        for rel in ["status.json", "STATUS.txt", "env.txt", "conversion_images/manifest.csv",
                    ] + CODE_FILES:
            if os.path.exists(f"{RUN}/{rel}"):
                z.write(f"{RUN}/{rel}", rel)
    os.replace(tmp, OUT_ZIP)

def run_step(name, cmd, est_min):
    elapsed_h = (time.time() - T_START) / 3600
    if elapsed_h > DEADLINE_H or elapsed_h + est_min / 60 > HARD_END_H:
        return {"step": name, "status": "SKIPPED (time budget)", "returncode": None, "minutes": 0.0}
    cmd = [sys.executable if c == "python" else c for c in cmd]
    print(f"\n{'=' * 100}\n[{datetime.datetime.utcnow():%H:%M:%S}] START {name}  (elapsed {elapsed_h:.2f} h)\n{'=' * 100}", flush=True)
    t0 = time.time()
    with open(f"{RUN}/logs/{name}.log", "w", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace", bufsize=1)
        for line in p.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = p.wait()
    mins = (time.time() - t0) / 60
    print(f"[{name}] finished rc={rc} in {mins:.1f} min", flush=True)
    return {"step": name, "status": "OK" if rc == 0 else f"FAILED rc={rc}", "returncode": rc, "minutes": round(mins, 1)}

for name, cmd, est in STEPS:
    try:
        status.append(run_step(name, cmd, est))
    except Exception as e:           # never let one step's crash stop the notebook
        status.append({"step": name, "status": f"ERROR {type(e).__name__}: {e}"[:120], "returncode": None, "minutes": 0.0})
        print(f"[{name}] notebook-side error: {e}", flush=True)
    json.dump(status, open(f"{RUN}/status.json", "w"), indent=1)
    make_zip()                       # insurance: the zip is valid after every step
    print(f"zip refreshed after {name}: {os.path.getsize(OUT_ZIP) / 1e6:.1f} MB", flush=True)


In [ ]:
# CELL 6 -- FINAL: status table, completeness check, ONE output zip
lines = [f"{s['step']:<32} {s['status']:<26} {s['minutes']:>7} min" for s in status]
missing = [f for f in EXPECTED if not os.path.exists(f"{RUN}/{f}")]
lines += ["", f"total wall time: {(time.time() - T_START) / 3600:.2f} h",
          "MISSING expected files: " + (", ".join(missing) if missing else "none")]
open(f"{RUN}/STATUS.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

# ---- gate table (the point of this notebook) ----
print("\n" + "=" * 100 + "\nVERIFY GATE RESULTS\n" + "=" * 100)
all_pass = True
for m in ("qwen2.5-vl-3b", "internvl3-2b", "qwen2.5-vl-7b"):
    p = f"{RUN}/results/mech_e3/{m}/verify.json"
    if not os.path.exists(p):
        print(f"{m}: NO verify.json (step crashed before the gate could write it -- read its log)"); all_pass = False; continue
    v = json.load(open(p)); all_pass &= bool(v["pass"])
    print(f"\n{m}: GATE {'PASSED' if v['pass'] else 'FAILED'}")
    for k, c in v["checks"].items():
        tag = ("PASS" if c["pass"] else "FAIL") if c.get("gating", True) else ("info:pass" if c["pass"] else "info:FAIL")
        info = {kk: vv for kk, vv in c.items() if kk not in ("pass", "gating")}
        print(f"  [{tag:<9}] {k}  {str(info)[:700]}")
    if "attn_impl" in v:
        print("     attention implementation per layer:", v["attn_impl"])
    for r in v.get("g2_breakdown", []):
        print("     G2 breakdown:", r)
print("\n>>> ALL GATES PASSED -> OK to launch the full re-run." if all_pass else "\n>>> NOT all gates passed -> do NOT launch the full re-run; send this output back.")
make_zip()
with zipfile.ZipFile(OUT_ZIP) as z:
    assert z.testzip() is None, "zip is corrupt"
    names = z.namelist()
print(f"\nOUTPUT: {OUT_ZIP}  ({os.path.getsize(OUT_ZIP) / 1e6:.1f} MB, {len(names)} files)")
for n in sorted(names):
    if n.startswith("results/") and n.endswith((".csv", ".json", ".png")):
        print("  ", n)
shutil.rmtree(RUN)      # leave ONLY the zip in /kaggle/working (the zip is verified above)
print("\nDONE." if not missing and all(s["status"] == "OK" for s in status) else
      "\nDONE WITH PROBLEMS -- read STATUS.txt inside the zip and the logs/ folder.")
